In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().parent

sys.path.append(str(project_root))

print(project_root)

C:\Users\AliyuAlbako\Desktop\healthguard-ai\backend


In [3]:
from src.preprocessing import (
    load_dataset,
    split_features_target,
    train_test_data,
    apply_smote,
    scale_data,
)

In [4]:
DATA_PATH = (
    project_root
    / "datasets"
    / "diabetes_012_health_indicators_BRFSS2015.csv"
)

df = load_dataset(DATA_PATH)

df.head()

,Diabetes_012,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,Fruits,...,AnyHealthcare,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income
0,0.0,1.0,1.0,1.0,40.0,1.0,0.0,0.0,0.0,0.0,...,1.0,0.0,5.0,18.0,15.0,1.0,0.0,9.0,4.0,3.0
1,0.0,0.0,0.0,0.0,25.0,1.0,0.0,0.0,1.0,0.0,...,0.0,1.0,3.0,0.0,0.0,0.0,0.0,7.0,6.0,1.0
2,0.0,1.0,1.0,1.0,28.0,0.0,0.0,0.0,0.0,1.0,...,1.0,1.0,5.0,30.0,30.0,1.0,0.0,9.0,4.0,8.0
3,0.0,1.0,0.0,1.0,27.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,2.0,0.0,0.0,0.0,0.0,11.0,3.0,6.0
4,0.0,1.0,1.0,1.0,24.0,0.0,0.0,0.0,1.0,1.0,...,1.0,0.0,2.0,3.0,0.0,0.0,0.0,11.0,5.0,4.0


In [5]:
X, y = split_features_target(df)

print(X.shape)
print(y.shape)

(253680, 21)
(253680,)


In [6]:
(
    X_train,
    X_test,
    y_train,
    y_test,
) = train_test_data(
    X,
    y,
)

In [7]:
print(X_train.shape)
print(X_test.shape)

(202944, 21)
(50736, 21)


### class imbalance checks

In [8]:
y_train.value_counts()


Diabetes_012
0.0    170962
2.0     28277
1.0      3705
Name: count, dtype: int64

## Apply SMOTE

In [9]:
X_train_smote, y_train_smote = apply_smote(
    X_train,
    y_train,
)

### classes aftet applying SMOTE

In [10]:
y_train_smote.value_counts()

Diabetes_012
0.0    170962
2.0    170962
1.0    170962
Name: count, dtype: int64

### Scale data

In [11]:
(
    scaler,
    X_train_scaled,
    X_test_scaled,
) = scale_data(
    X_train_smote,
    X_test,
)

In [12]:
from src.trainer import ModelTrainer

from src.evaluator import ModelEvaluator

In [13]:
trainer = ModelTrainer()

evaluator = ModelEvaluator()

In [14]:
results = []

In [15]:
for name in trainer.models.keys():

    print("="*60)

    print(name)

    print("="*60)

    if name == "Logistic Regression":

        model = trainer.train(

            name,

            X_train_scaled,

            y_train_smote

        )

        metrics = evaluator.evaluate(

            model,

            X_test_scaled,

            y_test

        )

    else:

        model = trainer.train(

            name,

            X_train_smote,

            y_train_smote

        )

        metrics = evaluator.evaluate(

            model,

            X_test,

            y_test

        )

    metrics["Model"] = name

    results.append(metrics)

Logistic Regression
Decision Tree
Random Forest
XGBoost


C:\Users\AliyuAlbako\Desktop\healthguard-ai\venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


LightGBM
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.140629 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 5355
[LightGBM] [Info] Number of data points in the train set: 512886, number of used features: 21
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612


C:\Users\AliyuAlbako\Desktop\healthguard-ai\venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [16]:
import pandas as pd

results_df = pd.DataFrame(results)

results_df

,Accuracy,Precision,Recall,F1,Model
0,0.641675,0.849142,0.641675,0.717367,Logistic Regression
1,0.769454,0.781910,0.769454,0.775487,Decision Tree
2,0.839503,0.795720,0.839503,0.809535,Random Forest
3,0.845297,0.802537,0.845297,0.815344,XGBoost
4,0.845356,0.804322,0.845356,0.817632,LightGBM


In [17]:
results_df.sort_values(

    by="F1",

    ascending=False

)

,Accuracy,Precision,Recall,F1,Model
4,0.845356,0.804322,0.845356,0.817632,LightGBM
3,0.845297,0.802537,0.845297,0.815344,XGBoost
2,0.839503,0.795720,0.839503,0.809535,Random Forest
1,0.769454,0.781910,0.769454,0.775487,Decision Tree
0,0.641675,0.849142,0.641675,0.717367,Logistic Regression


In [18]:
best_model_name = results_df.sort_values(

    by="F1",

    ascending=False

).iloc[0]["Model"]

print(best_model_name)

LightGBM


In [20]:
if best_model_name == "Logistic Regression":
    best_model = trainer.train(
        best_model_name,
        X_train_scaled,
        y_train_smote
    )
else:
    best_model = trainer.train(
        best_model_name,
        X_train_smote,
        y_train_smote
    )

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.069613 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 5355
[LightGBM] [Info] Number of data points in the train set: 512886, number of used features: 21
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612


In [21]:
trainer.save_model(

    best_model,

    "../models/best_model.pkl"

)

In [22]:
import joblib

joblib.dump(

    scaler,

    "../models/scaler.pkl"

)

['../models/scaler.pkl']